In [1]:
%pip install sqlite3

ERROR: Could not find a version that satisfies the requirement sqlite3 (from versions: none)
ERROR: No matching distribution found for sqlite3
Note: you may need to restart the kernel to use updated packages.


In [37]:
import pandas as pd
import sqlite3 

In [38]:
pd.set_option('display.max_columns', None)


In [4]:
# using sqlite3 sdk

connection = sqlite3.connect("./compas.db")

cursor = connection.cursor() 

data = cursor.execute("SELECT * FROM people")

ds = data.fetchall()

columns = [data.description[i][0] for i in range(len(data.description))]
len(columns)



41

In [39]:
# using pandas sdk
cnx = sqlite3.connect("./compas.db")

df = pd.read_sql_query("SELECT * FROM people", cnx)

In [40]:
# step (1) filter for only the relevant cols

cols = df.columns.to_list()

cols = ['id', 'name', 'first', 'last', 'sex', 'race', 'dob', 'age', 'age_cat', 'juv_fel_count', 'juv_misd_count',
         'juv_other_count', 'compas_screening_date', 'decile_score', 'score_text', 'violent_recid', 'priors_count', 
         'days_b_screening_arrest', 'c_jail_in', 'c_jail_out', 'c_case_number', 'c_days_from_compas', 'c_arrest_date', 
         'c_offense_date', 'c_charge_degree', 'c_charge_desc', 'is_recid', 'num_r_cases', 'r_case_number', 'r_charge_degree', 
         'r_days_from_arrest', 'r_offense_date', 'r_charge_desc', 'r_jail_in', 'r_jail_out', 'is_violent_recid', 'num_vr_cases', 
         'vr_case_number', 'vr_charge_degree', 'vr_offense_date', 'vr_charge_desc']

new_cols = ['sex', 'race', 'age_cat', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'c_charge_degree', 'c_charge_desc','is_recid']



In [ ]:
# observing prior count distribution
df['priors_count'].value_counts().sort_index().to_dict()

# observing juv counts distribution
juv_counts = df['juv_fel_count'] + df['juv_misd_count'] + df['juv_other_count']
juv_counts.value_counts().sort_index().to_dict()


{0: 10382,
 1: 808,
 2: 257,
 3: 142,
 4: 75,
 5: 42,
 6: 15,
 7: 8,
 8: 7,
 9: 6,
 10: 5,
 11: 2,
 14: 5,
 20: 1,
 21: 2}

In [ ]:
# step (2) now do some processing

def load_dataset(df):
    cols = ['sex','race','age_cat','juv_fel_count','juv_misd_count',
            'juv_other_count','priors_count','c_charge_degree','c_charge_desc','is_recid']
    df = df[cols].copy()

    felony      = {'(F1)','(F2)','(F3)','(F5)','(F6)','(F7)'}
    misdemeanor = {'(M1)','(M2)'}
    df = df[df['c_charge_degree'].isin(felony | misdemeanor)].copy()

    # drop rows missing the fields we actually use
    df = df.dropna(subset=['sex','race','age_cat','priors_count','is_recid'])

    out = pd.DataFrame(index=df.index)
    out['charge_degree'] = df['c_charge_degree'].map(
        lambda c: 'felony' if c in felony else 'misdemeanor')

    juv = df['juv_fel_count'] + df['juv_misd_count'] + df['juv_other_count']
    out['juv_counts'] = juv.map(lambda x: '1+' if x > 0 else '0')

    out['priors_bin'] = pd.cut(df['priors_count'], bins=[-1, 0, 3, 10, 50],
                               labels=['none','low','moderate','high'])

    out['sex']     = df['sex']
    out['race']    = df['race']
    out['age_cat'] = df['age_cat']
    out['is_recid'] = df['is_recid']
    return out

ds = load_dataset(df)



10920
10920


In [23]:
df['c_charge_degree']
felony      = {'(F1)','(F2)','(F3)','(F5)','(F6)','(F7)'}
misdemeanor = {'(M1)','(M2)'}


# make the charge degrees map to felony and misdemeanor, if element not in fleony or dismeanor dont keep
# task 1 learn how map func works

df = df[df['c_charge_degree'].isin(felony | misdemeanor)].copy()
df['charge_degree'] = df['c_charge_degree'].map(lambda c: 'felony' if c in felony else 'misdemeanor')
df['charge_degree']

0             felony
2             felony
3             felony
4             felony
5             felony
            ...     
11752         felony
11753    misdemeanor
11754    misdemeanor
11755    misdemeanor
11756         felony
Name: charge_degree, Length: 10920, dtype: object

In [ ]:
juv_counts = df['juv_fel_count'] + df['juv_misd_count'] + df['juv_other_count']




In [9]:
df[new_cols]

,sex,race,age_cat,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,c_charge_desc,is_recid
0,Male,Other,Greater than 45,0,0,0,0,(F3),Aggravated Assault w/Firearm,0
1,Male,Caucasian,25 - 45,0,0,0,0,None,None,-1
2,Male,African-American,25 - 45,0,0,0,0,(F3),Felony Battery w/Prior Convict,1
3,Male,African-American,Less than 25,0,0,1,4,(F3),Possession of Cocaine,1
4,Male,African-American,Less than 25,0,1,0,1,(F3),Possession of Cannabis,0
...,...,...,...,...,...,...,...,...,...,...
11752,Male,Other,Greater than 45,0,0,0,1,(F3),Burglary Structure Unoccup,0
11753,Male,Caucasian,Less than 25,0,3,5,3,(M1),Battery,1
11754,Male,Other,25 - 45,0,0,0,0,(M1),Battery,0
11755,Male,Caucasian,25 - 45,0,0,0,2,(M1),arrest case no charge,0


shape of people table:
rows: (11757) | cols: 41

relevant cols